# 🛡️ File Forensics
### Cybersecurity Application — File Format & Extension Mismatch Detector

---

| Field | Details |
|---|---|
| **Student Name** | Bhanupriya Mehra |
| **Project** | File Forensics |
| **Technology** | Python 3 · Flask · HTML · CSS · JavaScript |
| **Description** | Examines uploaded files to detect mismatches between filename extensions and actual binary formats using magic-byte signatures, entropy analysis, and risk scoring. |

---

## Table of Contents
1. [Project Overview](#1.-Project-Overview)
2. [Dependencies & Setup](#2.-Dependencies-&-Setup)
3. [Magic Byte Signature Table](#3.-Magic-Byte-Signature-Table)
4. [Extension → MIME Mapping](#4.-Extension-→-MIME-Mapping)
5. [Core Analysis Functions](#5.-Core-Analysis-Functions)
6. [Format Detection Engine](#6.-Format-Detection-Engine)
7. [Risk Assessment Engine](#7.-Risk-Assessment-Engine)
8. [Full File Analyzer](#8.-Full-File-Analyzer)
9. [Live Demonstrations](#9.-Live-Demonstrations)
10. [Flask Web Application](#10.-Flask-Web-Application)
11. [Test Suite](#11.-Test-Suite)
12. [Summary & Results](#12.-Summary-&-Results)

---
## 1. Project Overview

**File Forensics** is a fully functional web-based cybersecurity tool that identifies what a file *really is* — regardless of its declared extension.

### How it works
```
Upload File  →  Read Magic Bytes  →  Detect True Format  →  Compare Extension  →  Score Risk  →  Report
```

### Key Features
| Feature | Description |
|---|---|
| **Format Detection** | Reads binary magic bytes, refines ZIP/RIFF containers, applies text heuristics |
| **Mismatch Detection** | Compares declared extension vs. detected MIME type |
| **Risk Assessment** | Scores files 0–100 → SAFE / LOW / MEDIUM / HIGH |
| **Cryptographic Hashes** | MD5 + SHA-256 for every file |
| **Shannon Entropy** | Measures randomness to detect encryption/compression |
| **No Database** | All analysis in-memory; files deleted after analysis |

### Architecture
```
Browser (HTML/CSS/JS)
       │  multipart/form-data POST
       ▼
Flask  app.py  →  analyzer.py  →  JSON response
                      │
              ┌───────┴────────┐
         magic_bytes    text_heuristics
         zip_refine     riff_refine
         risk_score     hashes + entropy
```

---
## 2. Dependencies & Setup

In [ ]:
# Install required packages
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'flask', 'flask-cors', 'werkzeug', '--quiet'])
print('✅ Dependencies installed')

In [ ]:
# Standard library imports — no third-party analysis libraries needed
import os
import math
import hashlib
import zipfile
import tempfile
import json
from datetime import datetime
from pathlib import Path
from pprint import pprint

print('✅ All imports successful')
print(f'Python version: {sys.version}')

---
## 3. Magic Byte Signature Table

Every known binary format starts with a unique sequence of bytes called a **magic number** or **file signature**.
This table maps `(offset, byte_pattern) → (mime_type, label)` for 40+ formats.

| Format | Offset | Hex Signature | Label |
|---|---|---|---|
| JPEG | 0 | `FF D8 FF` | JPEG Image |
| PNG | 0 | `89 50 4E 47 0D 0A 1A 0A` | PNG Image |
| PDF | 0 | `25 50 44 46` (`%PDF`) | PDF Document |
| ZIP/DOCX | 0 | `50 4B 03 04` | ZIP / OOXML |
| Windows PE | 0 | `4D 5A` (`MZ`) | PE Executable |
| ELF | 0 | `7F 45 4C 46` | ELF Binary |
| MP3 | 0 | `49 44 33` (`ID3`) | MP3 Audio |

In [ ]:
# ─────────────────────────────────────────────────────────────
# SIGNATURES — (offset, bytes_pattern, mime_type, label)
# ─────────────────────────────────────────────────────────────
SIGNATURES = [
    # ── Images ──
    (0, b'\xff\xd8\xff',               'image/jpeg',                   'JPEG Image'),
    (0, b'\x89PNG\r\n\x1a\n',         'image/png',                    'PNG Image'),
    (0, b'GIF87a',                     'image/gif',                    'GIF Image'),
    (0, b'GIF89a',                     'image/gif',                    'GIF Image'),
    (0, b'BM',                         'image/bmp',                    'BMP Image'),
    (0, b'\x00\x00\x01\x00',          'image/x-icon',                 'ICO Icon'),
    (0, b'RIFF',                       None,                           'RIFF Container'),  # refined
    (0, b'\x49\x49\x2a\x00',          'image/tiff',                   'TIFF Image (LE)'),
    (0, b'\x4d\x4d\x00\x2a',          'image/tiff',                   'TIFF Image (BE)'),
    (0, b'\x00\x00\x00\x0cjP  ',      'image/jp2',                    'JPEG 2000'),

    # ── Documents ──
    (0, b'%PDF',                       'application/pdf',              'PDF Document'),
    (0, b'\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1', 'application/msoffice', 'MS Office / OLE2 Document'),
    (0, b'PK\x03\x04',                'application/zip',              'ZIP / OOXML / ODF'),  # refined

    # ── Audio / Video ──
    (0, b'ID3',                        'audio/mpeg',                   'MP3 Audio (ID3)'),
    (0, b'\xff\xfb',                   'audio/mpeg',                   'MP3 Audio'),
    (0, b'\xff\xf3',                   'audio/mpeg',                   'MP3 Audio'),
    (0, b'\xff\xf2',                   'audio/mpeg',                   'MP3 Audio'),
    (0, b'fLaC',                       'audio/flac',                   'FLAC Audio'),
    (0, b'OggS',                       'audio/ogg',                    'OGG Audio/Video'),
    (4, b'ftyp',                       'video/mp4',                    'MP4 / M4A / MOV'),
    (0, b'\x1a\x45\xdf\xa3',          'video/webm',                   'WebM / MKV'),

    # ── Archives ──
    (0, b'\x1f\x8b',                   'application/gzip',             'GZIP Archive'),
    (0, b'BZh',                        'application/x-bzip2',          'BZIP2 Archive'),
    (0, b'\xfd7zXZ\x00',              'application/x-xz',             'XZ Archive'),
    (0, b'7z\xbc\xaf\x27\x1c',        'application/x-7z-compressed',  '7-Zip Archive'),
    (0, b'Rar!\x1a\x07\x00',          'application/x-rar',            'RAR Archive'),
    (0, b'Rar!\x1a\x07\x01\x00',      'application/x-rar',            'RAR5 Archive'),
    (0, b'\xca\xfe\xba\xbe',          'application/java-archive',     'Java Class / Universal Binary'),

    # ── Executables ──
    (0, b'MZ',                         'application/x-msdownload',     'Windows PE Executable'),
    (0, b'\x7fELF',                    'application/x-elf',            'ELF Executable (Linux/Unix)'),
    (0, b'\xfe\xed\xfa\xce',          'application/x-mach-binary',    'Mach-O Executable (32-bit)'),
    (0, b'\xfe\xed\xfa\xcf',          'application/x-mach-binary',    'Mach-O Executable (64-bit)'),
    (0, b'\xce\xfa\xed\xfe',          'application/x-mach-binary',    'Mach-O Executable (32-bit LE)'),
    (0, b'\xcf\xfa\xed\xfe',          'application/x-mach-binary',    'Mach-O Executable (64-bit LE)'),

    # ── Fonts ──
    (0, b'wOFF',                       'font/woff',                    'WOFF Font'),
    (0, b'wOF2',                       'font/woff2',                   'WOFF2 Font'),
    (0, b'\x00\x01\x00\x00\x00',      'font/ttf',                     'TrueType Font'),
    (0, b'OTTO',                       'font/otf',                     'OpenType Font'),

    # ── Misc ──
    (0,   b'SQLite format 3\x00',      'application/x-sqlite3',        'SQLite Database'),
    (0,   b'\x50\x4b\x05\x06',        'application/zip',              'Empty ZIP'),
    (0,   b'\x25\x21',                 'application/postscript',       'PostScript'),
    (257, b'ustar\x00',               'application/x-tar',            'TAR Archive'),
    (257, b'ustar  \x00',             'application/x-tar',            'TAR Archive (GNU)'),
]

print(f'✅ Loaded {len(SIGNATURES)} file signatures')

# Preview first 10
print('\nSample signatures:')
print(f'{"Offset":<8} {"Pattern (hex)":<30} {"MIME Type":<35} {"Label"}')
print('-' * 100)
for offset, pattern, mime, label in SIGNATURES[:10]:
    hex_pat = pattern.hex() if pattern else 'N/A'
    print(f'{offset:<8} {hex_pat:<30} {str(mime):<35} {label}')

---
## 4. Extension → MIME Mapping

Each file extension maps to one or more valid MIME types.  
If the detected MIME is **not** in this list → **mismatch flagged**.

In [ ]:
EXTENSION_MAP = {
    # Images
    '.jpg':  ['image/jpeg'],
    '.jpeg': ['image/jpeg'],
    '.png':  ['image/png'],
    '.gif':  ['image/gif'],
    '.bmp':  ['image/bmp'],
    '.ico':  ['image/x-icon'],
    '.tif':  ['image/tiff'],
    '.tiff': ['image/tiff'],
    '.webp': ['image/webp'],
    '.jp2':  ['image/jp2'],
    '.svg':  ['image/svg+xml'],
    # Documents
    '.pdf':  ['application/pdf'],
    '.doc':  ['application/msoffice'],
    '.xls':  ['application/msoffice'],
    '.ppt':  ['application/msoffice'],
    '.docx': ['application/zip'],
    '.xlsx': ['application/zip'],
    '.pptx': ['application/zip'],
    '.odt':  ['application/zip'],
    '.ods':  ['application/zip'],
    '.odp':  ['application/zip'],
    '.rtf':  ['text/rtf', 'application/rtf'],
    # Audio
    '.mp3':  ['audio/mpeg'],
    '.flac': ['audio/flac'],
    '.ogg':  ['audio/ogg', 'video/ogg'],
    '.wav':  ['audio/wav', 'audio/x-wav'],
    '.m4a':  ['video/mp4', 'audio/mp4'],
    '.aac':  ['audio/aac'],
    # Video
    '.mp4':  ['video/mp4'],
    '.mov':  ['video/mp4', 'video/quicktime'],
    '.mkv':  ['video/webm'],
    '.webm': ['video/webm'],
    '.avi':  ['video/avi', 'video/x-msvideo'],
    # Archives
    '.zip':  ['application/zip'],
    '.gz':   ['application/gzip'],
    '.bz2':  ['application/x-bzip2'],
    '.xz':   ['application/x-xz'],
    '.7z':   ['application/x-7z-compressed'],
    '.rar':  ['application/x-rar'],
    '.tar':  ['application/x-tar'],
    '.jar':  ['application/java-archive', 'application/zip'],
    # Executables
    '.exe':  ['application/x-msdownload'],
    '.dll':  ['application/x-msdownload'],
    '.elf':  ['application/x-elf'],
    # Text / Code
    '.txt':  ['text/plain'],
    '.csv':  ['text/plain', 'text/csv'],
    '.html': ['text/html', 'text/plain'],
    '.htm':  ['text/html', 'text/plain'],
    '.xml':  ['text/xml', 'application/xml', 'text/plain'],
    '.json': ['application/json', 'text/plain'],
    '.js':   ['application/javascript', 'text/plain'],
    '.css':  ['text/css', 'text/plain'],
    '.py':   ['text/x-python', 'text/plain'],
    '.sh':   ['application/x-sh', 'text/plain'],
    '.bat':  ['application/x-bat', 'text/plain'],
    # Fonts
    '.ttf':  ['font/ttf'],
    '.otf':  ['font/otf'],
    '.woff': ['font/woff'],
    '.woff2':['font/woff2'],
    # Database
    '.sqlite': ['application/x-sqlite3'],
    '.db':     ['application/x-sqlite3'],
    # Misc
    '.ps':  ['application/postscript'],
    '.swf': ['application/x-shockwave-flash'],
}

# Risk classification sets
HIGH_RISK_TYPES = {
    'application/x-msdownload', 'application/x-elf',
    'application/x-mach-binary', 'application/x-sh', 'application/x-bat',
}
MEDIUM_RISK_TYPES = {
    'application/java-archive', 'application/x-7z-compressed',
    'application/x-rar', 'application/zip', 'application/gzip',
    'application/x-bzip2', 'application/x-xz', 'application/x-tar',
    'application/x-sqlite3',
}

print(f'✅ Extension map loaded: {len(EXTENSION_MAP)} extensions')
print(f'   HIGH risk types  : {len(HIGH_RISK_TYPES)}')
print(f'   MEDIUM risk types: {len(MEDIUM_RISK_TYPES)}')

---
## 5. Core Analysis Functions

Low-level helpers used by the detection engine.

In [ ]:
def _read_header(file_path: str, size: int = 512) -> bytes:
    """Read the first `size` bytes of a file (the magic-byte header)."""
    with open(file_path, 'rb') as f:
        return f.read(size)


def _compute_hashes(file_path: str) -> dict:
    """Compute MD5 and SHA-256 digests of the full file content."""
    md5    = hashlib.md5()
    sha256 = hashlib.sha256()
    with open(file_path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            md5.update(chunk)
            sha256.update(chunk)
    return {'md5': md5.hexdigest(), 'sha256': sha256.hexdigest()}


def _compute_entropy(file_path: str) -> float:
    """
    Compute Shannon entropy (0–8 bits/byte).
    Formula: H = -Σ p(x) · log2(p(x))
    High entropy (>7.0) suggests encryption or strong compression.
    """
    counts = [0] * 256
    total  = 0
    with open(file_path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            for byte in chunk:
                counts[byte] += 1
            total += len(chunk)
    if total == 0:
        return 0.0
    entropy = 0.0
    for c in counts:
        if c:
            p = c / total
            entropy -= p * math.log2(p)
    return round(entropy, 4)


def _is_text(header: bytes) -> bool:
    """Return True if >85% of header bytes are printable ASCII — likely a text file."""
    if not header:
        return False
    printable = sum(1 for b in header if 0x09 <= b <= 0x0d or 0x20 <= b <= 0x7e)
    return (printable / len(header)) > 0.85


def _detect_text_subtype(header: bytes) -> tuple:
    """Classify text files into HTML, XML, JSON, Python, Shell, CSS, JS, CSV, or plain text."""
    try:
        text = header.decode('utf-8', errors='replace').lstrip('\ufeff').lstrip()
    except Exception:
        return 'text/plain', 'Plain Text'

    stripped = text.strip()
    first_line = stripped.split('\n')[0]

    if stripped.startswith('<?xml'):
        return 'text/xml', 'XML Document'
    if stripped.startswith('<!DOCTYPE html') or stripped.lower().startswith('<html'):
        return 'text/html', 'HTML Document'
    if stripped.startswith('{') or stripped.startswith('['):
        return 'application/json', 'JSON Data'
    if stripped.startswith('#!'):
        if 'python' in first_line:
            return 'text/x-python', 'Python Script'
        if 'bash' in first_line or 'sh' in first_line:
            return 'application/x-sh', 'Shell Script'
    if 'import ' in text or 'def ' in text or 'class ' in text:
        return 'text/x-python', 'Python Script'
    if stripped.startswith('function') or 'const ' in stripped or 'let ' in stripped:
        return 'application/javascript', 'JavaScript'
    if '{' in stripped and ':' in stripped:
        return 'text/css', 'CSS Stylesheet'
    if first_line.count(',') >= 2:
        return 'text/csv', 'CSV Data'
    return 'text/plain', 'Plain Text'


def _refine_zip(file_path: str) -> tuple:
    """Look inside ZIP to identify DOCX / XLSX / PPTX / ODF / JAR."""
    try:
        with zipfile.ZipFile(file_path, 'r') as z:
            names = z.namelist()
        if 'word/document.xml'      in names: return 'application/zip', 'DOCX (Word Document)'
        if 'xl/workbook.xml'        in names: return 'application/zip', 'XLSX (Excel Spreadsheet)'
        if 'ppt/presentation.xml'   in names: return 'application/zip', 'PPTX (PowerPoint Presentation)'
        if any(n.startswith('META-INF/') for n in names):
            if any(n.endswith('.class') for n in names):
                return 'application/java-archive', 'JAR (Java Archive)'
            return 'application/zip', 'ODF / JAR Package'
        if 'mimetype' in names: return 'application/zip', 'ODF Document'
        return 'application/zip', 'ZIP Archive'
    except Exception:
        return 'application/zip', 'ZIP Archive'


def _refine_riff(header: bytes) -> tuple:
    """Distinguish WAV / AVI / WebP inside a RIFF container."""
    if len(header) >= 12:
        tag = header[8:12]
        if tag == b'WAVE': return 'audio/wav',   'WAV Audio'
        if tag == b'AVI ': return 'video/avi',   'AVI Video'
        if tag == b'WEBP': return 'image/webp',  'WebP Image'
    return 'application/octet-stream', 'RIFF Container'


def _human_size(size: int) -> str:
    """Convert bytes to a human-readable string (B, KB, MB, GB)."""
    for unit in ('B', 'KB', 'MB', 'GB'):
        if size < 1024:
            return f'{size:.1f} {unit}'
        size /= 1024
    return f'{size:.1f} TB'


print('✅ Core helper functions defined')

---
## 6. Format Detection Engine

The main detection pipeline:
1. Read first 512 bytes
2. Walk signature table → match magic bytes
3. If RIFF → refine by sub-type tag
4. If ZIP  → inspect entry list for DOCX/XLSX/PPTX/JAR
5. If no binary match → text heuristic

In [ ]:
def detect_format(file_path: str) -> dict:
    """
    Detect the actual format of a file by reading its binary content.

    Returns:
        dict with keys: mime_type, label, method
        method is one of: 'magic_bytes', 'text_heuristic', 'unknown', 'empty'
    """
    header = _read_header(file_path, 512)

    if not header:
        return {'mime_type': 'application/octet-stream', 'label': 'Empty File', 'method': 'empty'}

    # Walk the signature table
    for offset, pattern, mime, label in SIGNATURES:
        end = offset + len(pattern)
        if len(header) >= end and header[offset:end] == pattern:
            if mime is None and pattern == b'RIFF':
                mime, label = _refine_riff(header)
            elif mime == 'application/zip':
                mime, label = _refine_zip(file_path)
            return {'mime_type': mime, 'label': label, 'method': 'magic_bytes'}

    # Text fallback
    if _is_text(header):
        mime, label = _detect_text_subtype(header)
        return {'mime_type': mime, 'label': label, 'method': 'text_heuristic'}

    return {'mime_type': 'application/octet-stream', 'label': 'Unknown Binary', 'method': 'unknown'}


print('✅ detect_format() defined')

---
## 7. Risk Assessment Engine

Scores each file 0–100 using four weighted factors:

| Factor | Points |
|---|---|
| Extension mismatch detected | +50 |
| Detected type is executable / script | +40 |
| Detected type is archive / database | +20 |
| Dangerous disguise (PE/ELF hidden as .pdf/.jpg) | +10 |

**Score → Level:** `0–9 = SAFE` · `10–29 = LOW` · `30–59 = MEDIUM` · `60–100 = HIGH`

In [ ]:
def assess_risk(detected_mime: str, extension: str, mismatch: bool) -> dict:
    """
    Compute a 0-100 risk score and map it to a level.

    Returns:
        dict with keys: level (str), score (int), reasons (list[str])
    """
    reasons = []
    score   = 0

    # Factor 1 — mismatch
    if mismatch:
        score += 50
        reasons.append('File extension does not match detected format')

    # Factor 2 — format risk
    if detected_mime in HIGH_RISK_TYPES:
        score += 40
        reasons.append(f'Detected format ({detected_mime}) is an executable or script')
    elif detected_mime in MEDIUM_RISK_TYPES:
        score += 20
        reasons.append(f'Detected format ({detected_mime}) is an archive or database')

    # Factor 3 — dangerous disguise
    DANGEROUS = {
        ('application/x-msdownload', '.pdf'),  ('application/x-msdownload', '.jpg'),
        ('application/x-msdownload', '.docx'), ('application/x-msdownload', '.txt'),
        ('application/x-elf',        '.pdf'),  ('application/x-elf',        '.txt'),
        ('application/x-mach-binary','.pdf'),  ('application/x-sh',         '.txt'),
        ('application/x-sh',         '.jpg'),
    }
    if (detected_mime, extension.lower()) in DANGEROUS:
        score += 10
        reasons.append('High-risk format disguised as a benign file type')

    score = min(score, 100)

    # Map score → level
    if   score >= 60: level = 'HIGH'
    elif score >= 30: level = 'MEDIUM'
    elif score >= 10: level = 'LOW'
    else:             level = 'SAFE'

    if not reasons:
        reasons.append('No anomalies detected')

    return {'level': level, 'score': score, 'reasons': reasons}


print('✅ assess_risk() defined')

---
## 8. Full File Analyzer

Orchestrates all steps: format detection → mismatch check → risk scoring → hashes → entropy → report.

In [ ]:
def analyze_file(file_path: str, original_filename: str) -> dict:
    """
    Full forensic analysis of a single file.

    Args:
        file_path:         Absolute path to the file on disk.
        original_filename: Original name as provided by the user (preserves extension).

    Returns:
        Structured dict containing all analysis results.
    """
    name, ext = os.path.splitext(original_filename)
    ext       = ext.lower()
    file_size = os.path.getsize(file_path)

    # ── Step 1: Detect actual format ──────────────────────────
    fmt            = detect_format(file_path)
    detected_mime  = fmt['mime_type']
    detected_label = fmt['label']
    method         = fmt['method']

    # ── Step 2: Check extension match ─────────────────────────
    expected_mimes = EXTENSION_MAP.get(ext, [])
    if not ext:
        mismatch        = False
        mismatch_detail = 'No extension present'
    elif not expected_mimes:
        mismatch        = False
        mismatch_detail = f'Extension "{ext}" is not in the known-extensions list'
    else:
        mismatch = detected_mime not in expected_mimes
        if mismatch:
            mismatch_detail = (
                f'Extension "{ext}" expected {expected_mimes}, '
                f'but actual format is "{detected_mime}"'
            )
        else:
            mismatch_detail = 'Extension matches detected format'

    # ── Step 3: Risk scoring ───────────────────────────────────
    risk = assess_risk(detected_mime, ext, mismatch)

    # ── Step 4: Cryptographic hashes ──────────────────────────
    hashes  = _compute_hashes(file_path)

    # ── Step 5: Shannon entropy ────────────────────────────────
    entropy = _compute_entropy(file_path)
    if   entropy > 7.2: entropy_note = 'Very high — may be encrypted or compressed'
    elif entropy > 6.0: entropy_note = 'High — may be compressed or contain encrypted sections'
    elif entropy > 4.0: entropy_note = 'Moderate — typical for mixed binary/text content'
    else:               entropy_note = 'Low — typical for text or structured data'

    return {
        'filename':         original_filename,
        'extension':        ext if ext else '(none)',
        'file_size':        file_size,
        'file_size_human':  _human_size(file_size),
        'detected_mime':    detected_mime,
        'detected_label':   detected_label,
        'detection_method': method,
        'expected_mimes':   expected_mimes,
        'mismatch':         mismatch,
        'mismatch_detail':  mismatch_detail,
        'risk':             risk,
        'hashes':           hashes,
        'entropy':          entropy,
        'entropy_note':     entropy_note,
        'analyzed_at':      datetime.utcnow().isoformat() + 'Z',
    }


print('✅ analyze_file() defined — full pipeline ready')

---
## 9. Live Demonstrations

We create real synthetic binary files in a temp directory and run the full analyzer on each.

In [ ]:
def make_file(content: bytes, filename: str) -> str:
    """Write content to a temp file and return the path."""
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=os.path.splitext(filename)[1])
    tmp.write(content)
    tmp.close()
    return tmp.name


def print_report(result: dict):
    """Pretty-print an analysis result."""
    risk = result['risk']
    level_icons = {'SAFE': '🟢', 'LOW': '🟡', 'MEDIUM': '🟠', 'HIGH': '🔴'}
    icon = level_icons.get(risk['level'], '⚪')
    mismatch_icon = '⚠️  MISMATCH' if result['mismatch'] else '✅  MATCH'

    print(f"{'─'*60}")
    print(f"  File     : {result['filename']}")
    print(f"  Extension: {result['extension']}")
    print(f"  Detected : {result['detected_label']} ({result['detected_mime']})")
    print(f"  Method   : {result['detection_method']}")
    print(f"  Status   : {mismatch_icon}")
    print(f"  Risk     : {icon} {risk['level']}  ({risk['score']}/100)")
    for r in risk['reasons']:
        print(f"             › {r}")
    print(f"  Entropy  : {result['entropy']}  — {result['entropy_note']}")
    print(f"  MD5      : {result['hashes']['md5']}")
    print(f"  SHA-256  : {result['hashes']['sha256']}")
    print(f"  Size     : {result['file_size_human']}")
    print(f"{'─'*60}")


print('✅ Demo helpers ready')

In [ ]:
# ── Demo 1: Legitimate JPEG ────────────────────────────────────
print('\n=== Demo 1: Legitimate JPEG file with .jpg extension ===')
p = make_file(b'\xff\xd8\xff\xe0' + b'\x00' * 200, 'photo.jpg')
result = analyze_file(p, 'photo.jpg')
print_report(result)
os.unlink(p)

In [ ]:
# ── Demo 2: Windows PE disguised as .txt ───────────────────────
print('\n=== Demo 2: Windows PE Executable disguised as readme.txt ===')
p = make_file(b'MZ' + b'\x00' * 200, 'readme.txt')
result = analyze_file(p, 'readme.txt')
print_report(result)
os.unlink(p)

In [ ]:
# ── Demo 3: PNG disguised as .pdf ──────────────────────────────
print('\n=== Demo 3: PNG Image disguised as invoice.pdf ===')
p = make_file(b'\x89PNG\r\n\x1a\n' + b'\x00' * 200, 'invoice.pdf')
result = analyze_file(p, 'invoice.pdf')
print_report(result)
os.unlink(p)

In [ ]:
# ── Demo 4: ELF Binary disguised as .jpg ───────────────────────
print('\n=== Demo 4: ELF Binary disguised as photo.jpg ===')
p = make_file(b'\x7fELF' + b'\x00' * 200, 'photo.jpg')
result = analyze_file(p, 'photo.jpg')
print_report(result)
os.unlink(p)

In [ ]:
# ── Demo 5: Legitimate PDF ─────────────────────────────────────
print('\n=== Demo 5: Legitimate PDF file ===')
p = make_file(b'%PDF-1.4 sample content' + b'\x00' * 100, 'report.pdf')
result = analyze_file(p, 'report.pdf')
print_report(result)
os.unlink(p)

In [ ]:
# ── Demo 6: Python script with correct .py extension ───────────
print('\n=== Demo 6: Python script with .py extension ===')
code = b'#!/usr/bin/env python3\nimport os\nprint("Hello")\n'
p = make_file(code, 'script.py')
result = analyze_file(p, 'script.py')
print_report(result)
os.unlink(p)

In [ ]:
# ── Demo 7: GZIP archive disguised as .jpg ─────────────────────
print('\n=== Demo 7: GZIP Archive disguised as image.jpg ===')
p = make_file(b'\x1f\x8b\x08' + b'\x00' * 200, 'image.jpg')
result = analyze_file(p, 'image.jpg')
print_report(result)
os.unlink(p)

---
## 10. Flask Web Application

The complete Flask backend — serves the frontend and exposes two API endpoints.

| Endpoint | Method | Description |
|---|---|---|
| `/` | GET | Serves `static/index.html` |
| `/static/<path>` | GET | Serves CSS, JS, assets |
| `/api/analyze` | POST | Accepts file uploads, returns JSON analysis |
| `/api/health` | GET | Server health check |

In [ ]:
# app.py — Flask backend (shown here for reference)
FLASK_APP_CODE = '''
import os, uuid, tempfile
from flask import Flask, request, jsonify, send_from_directory
from flask_cors import CORS
from werkzeug.utils import secure_filename
from analyzer import analyze_file

app = Flask(__name__, static_folder="static")
CORS(app)
app.config["MAX_CONTENT_LENGTH"] = 50 * 1024 * 1024   # 50 MB
UPLOAD_FOLDER = tempfile.mkdtemp(prefix="file_forensics_")

@app.route("/")
def index():
    return send_from_directory("static", "index.html")

@app.route("/static/<path:path>")
def serve_static(path):
    return send_from_directory("static", path)

@app.route("/api/analyze", methods=["POST"])
def analyze():
    if "files" not in request.files:
        return jsonify({"error": "No files provided"}), 400
    results, errors = [], []
    for f in request.files.getlist("files"):
        if not f.filename:
            continue
        safe = secure_filename(f.filename) or "upload"
        path = os.path.join(UPLOAD_FOLDER, f"{uuid.uuid4().hex}_{safe}")
        try:
            f.save(path)
            results.append(analyze_file(path, f.filename))
        except Exception as e:
            errors.append({"filename": f.filename, "error": str(e)})
        finally:
            if os.path.exists(path):
                os.remove(path)   # always delete temp file
    if not results and errors:
        return jsonify({"error": "All files failed", "details": errors}), 500
    return jsonify({"results": results, "errors": errors})

@app.route("/api/health")
def health():
    return jsonify({"status": "ok", "service": "File Forensics"})

if __name__ == "__main__":
    app.run(debug=True, host="127.0.0.1", port=5000)
'''

print('Flask app.py contents:')
print(FLASK_APP_CODE)

In [ ]:
# ── Test Flask routes using the test client (no server needed) ──
import sys
sys.path.insert(0, '.')

try:
    from app import app as flask_app
    import io

    with flask_app.test_client() as client:
        # Health check
        r = client.get('/api/health')
        print('GET /api/health :', r.status_code, r.get_json())

        # Upload a real JPEG
        data = {'files': (io.BytesIO(b'\xff\xd8\xff\xe0' + b'\x00'*200), 'test.jpg')}
        r2 = client.post('/api/analyze', data=data, content_type='multipart/form-data')
        res = r2.get_json()['results'][0]
        print(f'POST /api/analyze (test.jpg):')
        print(f'  detected : {res["detected_mime"]}')
        print(f'  mismatch : {res["mismatch"]}')
        print(f'  risk     : {res["risk"]["level"]}')

        # Upload PE disguised as .docx
        data2 = {'files': (io.BytesIO(b'MZ' + b'\x00'*200), 'malware.docx')}
        r3 = client.post('/api/analyze', data=data2, content_type='multipart/form-data')
        res3 = r3.get_json()['results'][0]
        print(f'POST /api/analyze (malware.docx):')
        print(f'  detected : {res3["detected_mime"]}')
        print(f'  mismatch : {res3["mismatch"]}')
        print(f'  risk     : {res3["risk"]["level"]}')

    print('\n✅ Flask route tests passed')

except ImportError:
    print('ℹ️  app.py not found in current directory — run this notebook from the project root.')

---
## 11. Test Suite

Automated tests covering all major detection and risk scenarios.

In [ ]:
def run_tests():
    """Run all unit tests and print a summary table."""

    TEST_CASES = [
        # (description,            content,                              filename,        exp_mime,                    exp_mismatch, exp_risk)
        ('JPEG match',             b'\xff\xd8\xff\xe0' + b'\x00'*100,   'photo.jpg',     'image/jpeg',                False,        'SAFE'),
        ('PNG match',              b'\x89PNG\r\n\x1a\n' + b'\x00'*100,  'icon.png',      'image/png',                 False,        'SAFE'),
        ('PDF match',              b'%PDF-1.4' + b'\x00'*100,           'report.pdf',    'application/pdf',           False,        'SAFE'),
        ('PE as .txt  (HIGH)',     b'MZ' + b'\x00'*100,                 'readme.txt',    'application/x-msdownload',  True,         'HIGH'),
        ('PE as .pdf  (HIGH)',     b'MZ' + b'\x00'*100,                 'invoice.pdf',   'application/x-msdownload',  True,         'HIGH'),
        ('PNG as .pdf (MEDIUM)',   b'\x89PNG\r\n\x1a\n' + b'\x00'*100, 'invoice.pdf',   'image/png',                 True,         'MEDIUM'),
        ('ELF as .jpg (HIGH)',     b'\x7fELF' + b'\x00'*100,           'photo.jpg',     'application/x-elf',         True,         'HIGH'),
        ('GZIP as .jpg (MEDIUM)',  b'\x1f\x8b\x08' + b'\x00'*100,      'image.jpg',     'application/gzip',          True,         'MEDIUM'),
        ('Plain text match',       b'Hello world plain text content.',   'notes.txt',     'text/plain',                False,        'SAFE'),
        ('Python script match',    b'#!/usr/bin/env python3\nimport os', 'script.py',     'text/x-python',             False,        'SAFE'),
        ('Unknown extension',      b'\xff\xd8\xff' + b'\x00'*100,       'file.unknownx', 'image/jpeg',                False,        'SAFE'),
        ('No extension',           b'%PDF-1.4' + b'\x00'*50,            'document',      'application/pdf',           False,        'SAFE'),
    ]

    passed = 0
    failed = 0
    rows   = []

    print(f"{'Test':<30} {'Detected MIME':<35} {'Mismatch':<10} {'Risk':<8} {'Status'}")
    print('─' * 100)

    for desc, content, filename, exp_mime, exp_mismatch, exp_risk in TEST_CASES:
        p = make_file(content, filename)
        try:
            r = analyze_file(p, filename)
            ok = (
                r['detected_mime'] == exp_mime and
                r['mismatch']      == exp_mismatch and
                r['risk']['level'] == exp_risk
            )
            if ok:
                passed += 1
                status = '✅ PASS'
            else:
                failed += 1
                status = f'❌ FAIL (got mime={r["detected_mime"]}, mismatch={r["mismatch"]}, risk={r["risk"]["level"]})'
            print(f"{desc:<30} {r['detected_mime']:<35} {str(r['mismatch']):<10} {r['risk']['level']:<8} {status}")
        finally:
            os.unlink(p)

    print('─' * 100)
    print(f'\n Results: {passed} passed · {failed} failed out of {len(TEST_CASES)} tests')
    if failed == 0:
        print('🎉 All tests passed!')


run_tests()

---
## 12. Summary & Results

### What this notebook demonstrates

| Component | Status |
|---|---|
| Magic byte signature table (40+ formats) | ✅ |
| RIFF container refinement (WAV/AVI/WebP) | ✅ |
| ZIP container refinement (DOCX/XLSX/PPTX/JAR) | ✅ |
| Text content heuristics (HTML/XML/JSON/Python/Shell/CSS/JS/CSV) | ✅ |
| Extension mismatch detection | ✅ |
| Risk scoring (0–100 → SAFE/LOW/MEDIUM/HIGH) | ✅ |
| MD5 + SHA-256 hash computation | ✅ |
| Shannon entropy computation | ✅ |
| Flask API endpoints (/api/analyze, /api/health) | ✅ |
| 12 automated test cases | ✅ |

### Risk Scoring Summary

| Scenario | Score | Level |
|---|---|---|
| Clean JPEG, PNG, PDF, text | 0 | 🟢 SAFE |
| PNG disguised as .pdf | 50 | 🟠 MEDIUM |
| ELF disguised as .jpg | 90 | 🔴 HIGH |
| PE disguised as .txt | 100 | 🔴 HIGH |
| GZIP disguised as .jpg | 70 | 🔴 HIGH |

### How to Run the Web App
```bash
cd "File Forensics"
pip install -r requirements.txt
python app.py
# Open http://127.0.0.1:5000
```

---
*File Forensics — Bhanupriya Mehra*